# LangGraph Advanced: Memory, Interrupts & Streaming

This notebook builds on the previous tutorials to explore LangGraph's advanced features:

1. **Short-Term Memory** - Checkpointers for conversation persistence
2. **Human-in-the-Loop (HITL)** - Interrupts for human approval/input
3. **Long-Term Memory** - Episodic (user facts) + Semantic (agent learnings) memory
4. **Streaming** - Real-time updates from graph execution

---

In [1]:
import os
import uuid
from dotenv import load_dotenv
from typing import Literal, Annotated

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv()

model = ChatOpenAI(model="gpt-5-mini", temperature=0)

---
# Part 1: Short-Term Memory (Checkpointer)

**Checkpointers** save snapshots of the graph state at each super-step. This enables:
- **Conversation persistence** - Resume conversations across invocations
- **Time-travel debugging** - Replay from any checkpoint
- **Fault tolerance** - Recover from failures mid-execution

### How It Works
- Checkpoints are saved automatically at each super-step
- Each **thread** (identified by `thread_id`) maintains its own checkpoint history
- For production, use `PostgresSaver`, `RedisSaver`, etc. instead of `InMemorySaver`

### Key Methods
| Method | Purpose |
|--------|--------|
| `get_state(config)` | Get the latest state snapshot |
| `get_state_history(config)` | Get all checkpoints for a thread |
| `update_state(config, values)` | Manually update state (for forking) |

### Building a ReAct Agent with Checkpointing

Now we compile with a **checkpointer**. The key addition is `builder.compile(checkpointer=checkpointer)` which enables state persistence across invocations.

In [2]:
from langgraph.checkpoint.memory import InMemorySaver

# Define tools
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"The weather in {city} is 72°F and sunny."

@tool  
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

tools = [get_weather, add_numbers]
model_with_tools = model.bind_tools(tools)

# Define state
class MessagesState(BaseModel):
    messages: Annotated[list[AnyMessage], add_messages] = Field(default_factory=list)

# Define nodes
def llm_call(state: MessagesState):
    return {
        "messages": [model_with_tools.invoke(
            [SystemMessage(content="You are a helpful assistant.")]
            + state.messages
        )]
    }

tool_node = ToolNode(tools)

def should_continue(state: MessagesState) -> Literal["tool_node", "__end__"]:
    if state.messages[-1].tool_calls:
        return "tool_node"
    return END

# Build graph
builder = StateGraph(MessagesState)
builder.add_node("llm_call", llm_call)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_call")
builder.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
builder.add_edge("tool_node", "llm_call")

# Compile WITH checkpointer for memory
checkpointer = InMemorySaver()
agent = builder.compile(checkpointer=checkpointer)

print("Agent compiled with checkpointer")

Agent compiled with checkpointer


### Visualize the Graph

Render the compiled graph as a mermaid diagram. This shows the flow: `START` → `llm_call` → (conditional) → `tool_node` or `END`.

In [3]:
from IPython.display import IFrame
import re
import base64
import urllib.parse

def display_mermaid(graph):
    """Display LangGraph as Mermaid diagram via mermaid.live embed."""
    mermaid_syntax = graph.get_graph().draw_mermaid()
    # Remove <p> tags that LangGraph 1.0.5+ adds
    mermaid_syntax = re.sub(r'<p>|</p>', '', mermaid_syntax)
    
    # Encode for mermaid.live URL
    graph_def = base64.urlsafe_b64encode(mermaid_syntax.encode()).decode()
    
    # Create standalone HTML with mermaid
    html_content = f"""<!DOCTYPE html>
<html><head>
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
</head><body>
<pre class="mermaid">{mermaid_syntax}</pre>
<script>mermaid.initialize({{startOnLoad:true}});</script>
</body></html>"""
    
    # Encode as data URL
    data_url = "data:text/html;base64," + base64.b64encode(html_content.encode()).decode()
    return IFrame(data_url, width=600, height=400)

display_mermaid(agent)

### Demo: Conversation Memory

With a checkpointer, each `thread_id` gets its own conversation history. The agent will remember what was said in previous turns within the same thread.

In [4]:
# Each thread maintains its own conversation history
# The thread_id is REQUIRED when using a checkpointer

config = {"configurable": {"thread_id": "user-123"}}

# First message
result = agent.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Jacob.")]},
    config=config
)
print("Turn 1:")
result["messages"][-1].pretty_print()

Turn 1:
================================== Ai Message ==================================

Hi Jacob — nice to meet you! I’m here to help. What can I do for you today?


In [5]:
# Second message - the agent remembers the first message!
result = agent.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config=config  # Same thread_id
)
print("Turn 2:")
result["messages"][-1].pretty_print()

Turn 2:
================================== Ai Message ==================================

Your name is Jacob.


The agent remembers the name from the previous turn because we used the same `thread_id`. Try changing the thread_id to see that different threads have separate memories.

### Inspecting Checkpoint History

Every super-step creates a checkpoint. Use `get_state_history()` to see all snapshots for a thread. Each checkpoint has a unique `checkpoint_id` you can use for time-travel.

In [6]:
# View all checkpoints (snapshots) for this thread
print("Checkpoint History:")
for i, state in enumerate(agent.get_state_history(config)):
    msg_count = len(state.values.get("messages", []))
    print(f"  [{i}] checkpoint_id={state.config['configurable']['checkpoint_id'][:8]}... messages={msg_count}")

Checkpoint History:
  [0] checkpoint_id=1f0e4d6e... messages=4
  [1] checkpoint_id=1f0e4d6e... messages=3
  [2] checkpoint_id=1f0e4d6e... messages=2
  [3] checkpoint_id=1f0e4d6e... messages=2
  [4] checkpoint_id=1f0e4d6e... messages=1
  [5] checkpoint_id=1f0e4d6e... messages=0


### Time-Travel: Forking from a Previous State

You can "go back in time" by specifying a `checkpoint_id`. This creates a **fork** - the original history is preserved, and you start a new branch from that point.

In [7]:
# Time-Travel: Replay from an earlier checkpoint
# This creates a FORK - original history is preserved

history = list(agent.get_state_history(config))
earlier_checkpoint = history[-2]  # Second-to-last state

# Fork from that checkpoint with a different message
fork_config = {
    "configurable": {
        "thread_id": "user-123",
        "checkpoint_id": earlier_checkpoint.config["configurable"]["checkpoint_id"]
    }
}

# Resume from the earlier state
result = agent.invoke(
    {"messages": [HumanMessage(content="Actually, call me Bob instead.")]},
    config=fork_config
)
print("Forked conversation:")
result["messages"][-1].pretty_print()

Forked conversation:
================================== Ai Message ==================================

Hi Bob — nice to meet you. How can I help you today?


---
# Part 2: Human-in-the-Loop (HITL) Interrupts

**Interrupts** pause graph execution to get human input. The graph state is saved, and execution resumes when you provide a response.

### How It Works
1. Graph execution reaches an `interrupt()` call
2. State is saved via the checkpointer
3. The interrupt value is returned under `__interrupt__`
4. You resume with `Command(resume=<value>)`, which becomes the `interrupt()` return value

### Use Cases
- **Approval workflows** - Human approves/rejects an action
- **Content review** - Human edits generated content
- **Tool approval** - Human approves sensitive tool calls (e.g., sending emails)
- **Input validation** - Loop until valid input is provided

> **Warning**: Don't put `interrupt()` inside try/except blocks, and don't let the order of interrupts change dynamically.

### Building an Approval Workflow

This example creates a workflow where:
1. `propose` node suggests an action
2. `approval_gate` pauses with `interrupt()` for human decision
3. Based on the response, routes to `execute` or `cancel`

The `interrupt()` call pauses the graph and returns a payload to the caller. When resumed with `Command(resume=...)`, the value becomes the return value of `interrupt()`.

In [8]:
from langgraph.types import Command, interrupt

# Example: Approval Workflow
# The agent proposes an action, human approves or rejects

class ApprovalState(BaseModel):
    action: str = ""
    status: str = "pending"

def propose_action(state: ApprovalState):
    """Agent proposes an action."""
    return {"action": "Send email to alice@example.com"}

def approval_gate(state: ApprovalState) -> Command[Literal["execute", "cancel"]]:
    """Pause for human approval."""
    
    # interrupt() pauses execution and returns this value to the caller
    # The value can be any JSON-serializable data to display in a UI
    decision = interrupt({
        "question": "Approve this action?",
        "proposed_action": state.action,
    })
    
    # When resumed, 'decision' contains whatever was passed to Command(resume=...)
    if decision.get("approved"):
        return Command(goto="execute")
    return Command(goto="cancel")

def execute_action(state: ApprovalState):
    return {"status": "executed"}

def cancel_action(state: ApprovalState):
    return {"status": "cancelled"}

# Build approval workflow
approval_builder = StateGraph(ApprovalState)
approval_builder.add_node("propose", propose_action)
approval_builder.add_node("approval_gate", approval_gate)
approval_builder.add_node("execute", execute_action)
approval_builder.add_node("cancel", cancel_action)

approval_builder.add_edge(START, "propose")
approval_builder.add_edge("propose", "approval_gate")
approval_builder.add_edge("execute", END)
approval_builder.add_edge("cancel", END)

approval_checkpointer = InMemorySaver()
approval_graph = approval_builder.compile(checkpointer=approval_checkpointer)

print("Approval workflow compiled")

Approval workflow compiled


### Starting the Workflow

When we invoke the graph, it runs until it hits the `interrupt()`. The result contains `__interrupt__` with the payload we passed to `interrupt()`. The graph is now paused, waiting for us to resume.

In [9]:
# Start the workflow - it will pause at the interrupt
config = {"configurable": {"thread_id": "approval-001"}}

result = approval_graph.invoke({"action": "", "status": "pending"}, config=config)

# The graph paused at the interrupt
print("Graph paused. Interrupt payload:")
print(result["__interrupt__"])

Graph paused. Interrupt payload:
[Interrupt(value={'question': 'Approve this action?', 'proposed_action': 'Send email to alice@example.com'}, id='8a9f1558efcbf33ed616895bde4eb10b')]


### Resuming with Approval

To resume, we call `invoke()` with `Command(resume=...)`. The value we pass becomes the return value of `interrupt()` in the paused node. Here we approve, so the graph routes to `execute`.

In [10]:
# Resume with approval
result = approval_graph.invoke(
    Command(resume={"approved": True}),
    config=config
)

print(f"Final status: {result['status']}")

Final status: executed


### Resuming with Rejection

Same workflow, but this time we reject. The graph routes to `cancel` instead. Note we use a different `thread_id` to start fresh.

In [11]:
# Try again with rejection
config2 = {"configurable": {"thread_id": "approval-002"}}

# Start workflow
result = approval_graph.invoke({"action": "", "status": "pending"}, config=config2)
print("Interrupt:", result["__interrupt__"])

# Resume with rejection
result = approval_graph.invoke(
    Command(resume={"approved": False}),
    config=config2
)
print(f"Final status: {result['status']}")

Interrupt: [Interrupt(value={'question': 'Approve this action?', 'proposed_action': 'Send email to alice@example.com'}, id='aa324de0b7f75076655a39054ed45b2e')]
Final status: cancelled


### Debug Interrupts

You can set interrupts at compile time for debugging without modifying node code.
This is useful for inspecting state at specific points in the graph.

In [12]:
# Compile with debug interrupts - pause before/after specific nodes
debug_graph = builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_before=["tool_node"],  # Pause before tool execution
    # interrupt_after=["llm_call"],  # Could also pause after LLM
)

config = {"configurable": {"thread_id": "debug-001"}}

# This will pause before tool_node
result = debug_graph.invoke(
    {"messages": [HumanMessage(content="What's the weather in Paris?")]},
    config=config
)

# Check current state
state = debug_graph.get_state(config)
print("Paused at:", state.next)  # Shows which node is next
print("Pending tool call:", state.values["messages"][-1].tool_calls)

Paused at: ('tool_node',)
Pending tool call: [{'name': 'get_weather', 'args': {'city': 'Paris'}, 'id': 'call_9EIFkqnPMXEQ3I36kIF2Pr1M', 'type': 'tool_call'}]


Compile with `interrupt_before=["tool_node"]` to pause before tool execution. This lets you inspect pending tool calls before they run - useful for debugging or adding approval for sensitive tools.

In [13]:
# Resume execution (pass None to continue from current state)
result = debug_graph.invoke(None, config=config)
print("Resumed and completed:")
result["messages"][-1].pretty_print()

Resumed and completed:
================================== Ai Message ==================================

It's 72°F and sunny in Paris.


Pass `None` as input to resume from the current state. The graph continues from where it paused.

---
# Part 3: Long-Term Memory with **Semantic Memory Only**

In this section we’ll use **Stores** as long-term memory that persists **across threads**.

We will store only **semantic memories**: stable facts about the user (and their constraints/preferences) that help personalize future conversations.

### Key Concepts
- **Namespace**: a tuple that organizes memories, e.g. `(user_id, "facts")`
- **Semantic search**: retrieves relevant facts via embeddings (`search(..., query=...)`)
- **put()**: write a memory (key/value)
- **search()**: retrieve memories (optionally via `query=...`)
- **get()**: retrieve a specific memory by key

### Pattern: Semantic-Memory Agent
1. User sends a message
2. Retrieve relevant **user facts** from `(user_id, "facts")`
3. Inject retrieved facts into the system prompt
4. Respond
5. Extract any **new** user facts and store them


### Setting Up a Semantic Memory Store

We’ll create one store with an embedding index and use it only for **user facts**:

- Namespace: `(user_id, "facts")`
- Value shape: `{"content": "<fact string>"}`

Each new conversation thread can reuse the same user’s facts by keeping `user_id` constant.


In [14]:
from langchain_openai import OpenAIEmbeddings
from langgraph.store.memory import InMemoryStore

# Semantic memory store (user facts only)
memory_store = InMemoryStore(
    index={
        "embed": OpenAIEmbeddings(model="text-embedding-3-small"),
        "dims": 1536,
    }
)

print("Semantic memory store ready (user facts only)")

Semantic memory store ready (user facts only)


### Part Y: Seed Extra Semantic Memories (for Retrieval Demo)

To make semantic retrieval obvious in the demos, we’ll pre-load a handful of user facts (some relevant, some irrelevant) into `(user_id, "facts")`.


In [15]:
import uuid

seed_user_id = "traveler"
seed_ns = (seed_user_id, "facts")

# Generic chatbot-style profile facts (not Spanish-specific)
seed_facts = [
    "User's name is Jordan.",
    "User is 34 years old.",
    "User based in Chicago.",
    "User works in consulting.",
    "User prefers concise responses in bullet points.",
    "User dislikes long theoretical explanations.",
    "User is usually available for 15–20 minute practice sessions.",
    "User prefers friendly, encouraging tone with gentle corrections.",
]

for fact in seed_facts:
    memory_store.put(seed_ns, str(uuid.uuid4()), {"content": fact})

print(f"Seeded {len(seed_facts)} semantic facts for user_id={seed_user_id!r}")

# Quick sanity check: retrieval should pick preference/goal facts
query = "keep it practical, concise bullets, gentle corrections"
hits = memory_store.search(seed_ns, query=query, limit=5)

Seeded 8 semantic facts for user_id='traveler'


### Building the Semantic-Memory Agent

This agent has three nodes:

1. `retrieve_facts` — semantic-search relevant user facts from the store
2. `respond` — answer using ONLY the retrieved facts (no guessing)
3. `extract_facts` — extract *new* user facts and store them for next time

For demo clarity, the assistant will append a short **“Memory used:”** line when it relies on stored facts.


In [16]:
import re
import uuid
from typing import Annotated

from pydantic import BaseModel, Field
from langchain_core.messages import AnyMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.store.base import BaseStore
from langgraph.types import RunnableConfig


class FactsExtraction(BaseModel):
    facts: list[str] = Field(
        default_factory=list,
        description="New, atomic facts explicitly stated by the user (no duplicates).",
    )


class SemanticAgentState(BaseModel):
    messages: Annotated[list[AnyMessage], add_messages] = Field(default_factory=list)
    retrieved_user_facts: list[str] = Field(default_factory=list)
    extracted_user_facts: list[str] = Field(default_factory=list)


def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip().lower())


def retrieve_facts(state: SemanticAgentState, config: RunnableConfig, *, store: BaseStore):
    query = state.messages[-1].content
    user_id = config["configurable"].get("user_id", "default")

    user_facts = [
        r.value["content"]
        for r in store.search((user_id, "facts"), query=query, limit=4)
    ]
    return {"retrieved_user_facts": user_facts}


def respond(state: SemanticAgentState, config: RunnableConfig):
    user_id = config["configurable"].get("user_id", "default")
    facts_block = "\n".join(f"- {f}" for f in state.retrieved_user_facts) or "(none)"

    system = f"""You are a helpful assistant who responds concisely.
Only use the memory facts listed below. If memory is empty, do not assume anything.

USER: {user_id}
SEMANTIC MEMORY (facts about this user):
{facts_block}

If you used any memory facts, end your response with:
Memory used: <comma-separated facts you relied on>
If you did not use memory facts, do not add that line."""
    return {"messages": [model.invoke([SystemMessage(content=system)] + state.messages)]}


def extract_facts(state: SemanticAgentState, config: RunnableConfig, *, store: BaseStore):
    user_id = config["configurable"].get("user_id", "default")
    user_ns = (user_id, "facts")

    existing_items = list(store.search(user_ns, limit=100))
    existing_facts = [it.value["content"] for it in existing_items]
    existing_norm = {_norm(f) for f in existing_facts}

    # After `respond`, the last message is the AI message; user message is right before it.
    user_msg = state.messages[-2].content if len(state.messages) >= 2 else state.messages[-1].content

    prompt = f"""Extract NEW facts about the user from the USER message.

Rules:
- Only extract facts the user explicitly stated about themselves, their constraints, or preferences.
- Facts must be atomic and reusable (one idea per fact).
- Do NOT include anything the assistant said.
- Do NOT duplicate existing facts (even paraphrased).
- Return 0-6 facts.
- Do not store the users requests / task asked of you as a memory.
- Phrase facts as i.e. "user is", not "I am"
- Keep wording of facts as close to original as possible and add context (i.e. if referrig to something state the full fact)

EXISTING FACTS:
{chr(10).join(f"- {f}" for f in existing_facts) or "(none)"}

USER MESSAGE:
{user_msg}"""

    result = model.with_structured_output(FactsExtraction).invoke(prompt)

    stored = []
    for fact in result.facts:
        if not fact:
            continue
        if _norm(fact) in existing_norm:
            continue
        store.put(user_ns, str(uuid.uuid4()), {"content": fact})
        existing_norm.add(_norm(fact))
        stored.append(fact)

    return {"extracted_user_facts": stored}


builder = StateGraph(SemanticAgentState)
builder.add_node("retrieve_facts", retrieve_facts)
builder.add_node("respond", respond)
builder.add_node("extract_facts", extract_facts)

builder.add_edge(START, "retrieve_facts")
builder.add_edge("retrieve_facts", "respond")
builder.add_edge("respond", "extract_facts")
builder.add_edge("extract_facts", END)

learning_agent = builder.compile(checkpointer=InMemorySaver(), store=memory_store)
print("Semantic-memory agent compiled (facts only)")

Semantic-memory agent compiled (facts only)


In [17]:
display_mermaid(learning_agent)

### Demo 1: First Interaction (Capture Preferences + Goal)

Example Set 1: Learning + preference memory

**Session 1**
First run: seed memory may not be useful yet, but the agent should *extract and store* stable user facts so future threads can personalize automatically.



In [18]:
config = {"configurable": {"thread_id": "spanish-1", "user_id": "traveler"}}

result = learning_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    """I’m trying to learn Spanish and I mainly want to be able to speak comfortably 
                    in real situations like airports, hotels, and restaurants.
                    I don’t enjoy studying grammar rules and textbooks.
                    Can you help me get started in a practical way?"""
                )
            )
        ]
    },
    config=config,
)

print("=" * 60)
print("MEMORY RETRIEVAL (before responding):")
print(f"  Semantic (facts): {result['retrieved_user_facts'] or '(first interaction)'}")
print("=" * 60)
print()
for msg in result["messages"]:
    msg.pretty_print()
print()
print("=" * 60)
print("MEMORY EXTRACTION (new facts stored):")
print(f"  New semantic facts: {result.get('extracted_user_facts') or '(none)'}")
print("=" * 60)


MEMORY RETRIEVAL (before responding):
  Semantic (facts): ['User is usually available for 15–20 minute practice sessions.', 'User prefers friendly, encouraging tone with gentle corrections.', 'User prefers concise responses in bullet points.', 'User dislikes long theoretical explanations.']

================================ Human Message =================================

I’m trying to learn Spanish and I mainly want to be able to speak comfortably 
                    in real situations like airports, hotels, and restaurants.
                    I don’t enjoy studying grammar rules and textbooks.
                    Can you help me get started in a practical way?
================================== Ai Message ==================================

Great — practical and conversation-focused is perfect. Quick plan and starter kit:

- What I’ll do
  - Short, spoken-focused lessons (no heavy grammar).
  - Friendly tone and gentle corrections.
  - 15–20 minute practice sessions you can fit int

### Demo 2: New Thread, Same User (Preference Carryover)

**Session 2** (new `thread_id`, same `user_id`)

**Memory payoff:** AI knows how to teach: 'User prefers friendly, encouraging tone with gentle corrections.', AI knows what to teach: 'user mainly wants to be able to speak comfortably in real situations like airports, hotels, and restaurants'


In [19]:
config = {"configurable": {"thread_id": "spanish-2", "user_id": "traveler"}}

result = learning_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    """I’m going to Spain and will need be able to speak Spanish.
                    Can you help me practice some realistic Spanish conversations I might actually have while traveling, 
                    and correct me if I make mistakes?"""
                )
            )
        ]
    },
    config=config,
)

print("=" * 60)
print("MEMORY RETRIEVAL (before responding):")
print(f"  Semantic (facts): {result['retrieved_user_facts'] or '(none)'}")
print("=" * 60)
print()
for msg in result["messages"]:
    msg.pretty_print()
print()
print("=" * 60)
print("MEMORY EXTRACTION (new facts stored):")
print(f"  New semantic facts: {result.get('extracted_user_facts') or '(none)'}")
print("=" * 60)


MEMORY RETRIEVAL (before responding):
  Semantic (facts): ['user mainly wants to be able to speak comfortably in real situations like airports, hotels, and restaurants', 'user is trying to learn Spanish', 'User is usually available for 15–20 minute practice sessions.', 'User prefers friendly, encouraging tone with gentle corrections.']

================================ Human Message =================================

I’m going to Spain and will need be able to speak Spanish.
                    Can you help me practice some realistic Spanish conversations I might actually have while traveling, 
                    and correct me if I make mistakes?
================================== Ai Message ==================================

¡Perfecto — puedo ayudarte! Podemos hacer breves sesiones de práctica (15–20 minutos) donde hagamos role-plays realistas (aeropuerto, hotel, restaurante). Yo te respondo como la otra persona en español, tú contestas, y te hago correcciones amables y breves con 

### Demo 3: Later Request (Uses Preferences Without Restating Them)

**Session 3** (another new `thread_id`, same `user_id`)

> I’ve been busy lately and need something structured. Can you design a 7-day study plan for me so I can practice Spanish a bit every day?

**Memory payoff:** AI knows how to schedule: 'User is usually available for 15–20 minute practice sessions.', and what to schedule lesson content on: 'user does not enjoy studying grammar rules and textbooks', 'user is going to Spain'

In [20]:
config = {"configurable": {"thread_id": "spanish-3", "user_id": "traveler"}}

result = learning_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    """I’ve been busy lately and need something structured. 
                    Can you design a 7-day study plan for me so I can practice Spanish a bit every day?"""
                )
            )
        ]
    },
    config=config,
)

print("=" * 60)
print("MEMORY RETRIEVAL (before responding):")
print(f"  Semantic (facts): {result['retrieved_user_facts'] or '(none)'}")
print("=" * 60)
print()
for msg in result["messages"]:
    msg.pretty_print()
print()
print("=" * 60)
print("MEMORY EXTRACTION (new facts stored):")
print(f"  New semantic facts: {result.get('extracted_user_facts') or '(none)'}")
print("=" * 60)


MEMORY RETRIEVAL (before responding):
  Semantic (facts): ['user is trying to learn Spanish', 'User is usually available for 15–20 minute practice sessions.', 'user does not enjoy studying grammar rules and textbooks', 'user is going to Spain']

================================ Human Message =================================

I’ve been busy lately and need something structured. 
                    Can you design a 7-day study plan for me so I can practice Spanish a bit every day?
================================== Ai Message ==================================

Great — here’s a compact, travel-focused 7-day plan you can do in 15–20 minutes a day. I avoided textbook-style grammar and kept each session practical and doable.

Day 1 — Essentials & pronunciation (15–20 min)
- Goal: Learn core survival phrases and basic sounds.
- 5 min: Quick warm-up — greetings (hola, buenos días, buenas noches), please/thank you (por favor, gracias), excuse me/sorry (perdón, lo siento).
- 10 min: Pronuncia

### Memory Viewer: Semantic Facts Only

Let’s inspect what’s stored for each user in `(user_id, "facts")`.


In [21]:
print("=== Semantic facts (by user) ===")
found = False

for ns in memory_store.list_namespaces():
    if len(ns) == 2 and ns[1] == "facts":
        user_id = ns[0]
        facts = memory_store.search(ns, limit=100)
        if facts:
            found = True
            print(f"\n{user_id}:")
            for item in facts:
                print(f"  - {item.value['content']}")

if not found:
    print("(no semantic facts stored yet)")


=== Semantic facts (by user) ===

traveler:
  - User's name is Jordan.
  - User is 34 years old.
  - User based in Chicago.
  - User works in consulting.
  - User prefers concise responses in bullet points.
  - User dislikes long theoretical explanations.
  - User is usually available for 15–20 minute practice sessions.
  - User prefers friendly, encouraging tone with gentle corrections.
  - user is trying to learn Spanish
  - user mainly wants to be able to speak comfortably in real situations like airports, hotels, and restaurants
  - user does not enjoy studying grammar rules and textbooks
  - user is going to Spain
  - user has been busy lately
  - user needs something structured
  - user wants to practice Spanish a bit every day


---
# Part 4: Streaming

Streaming provides real-time updates as the graph executes.
Different modes give different levels of detail.

### Stream Modes

| Mode | Description | Use Case |
|------|-------------|----------|
| `values` | Full state snapshot after each node | Debugging, state inspection |
| `updates` | Only the changes from each node | Progress tracking |
| `messages` | Token-by-token LLM output | Chat interfaces |
| `debug` | Maximum detail | Deep debugging |

### Sync vs Async
- `stream()` - Synchronous, blocks while streaming
- `astream()` - Async, allows concurrent operations

### Stream Mode: Updates

`stream_mode="updates"` yields only the **changes** from each node. Each chunk is a dict mapping node name to the updates it produced. Good for progress indicators.

In [22]:
# Stream mode: "updates" - See changes from each node
print("Stream mode: updates")
print("="*50)

config = {"configurable": {"thread_id": "stream-demo-1"}}

for chunk in agent.stream(
    {"messages": [HumanMessage(content="What's the weather in London?")]},
    config=config,
    stream_mode="updates"
):
    # Each chunk shows which node ran and what it changed
    for node_name, updates in chunk.items():
        print(f"\n[{node_name}] returned:")
        if "messages" in updates:
            for msg in updates["messages"]:
                content = msg.content[:100] if msg.content else f"tool_calls={msg.tool_calls}"
                print(f"  {msg.type}: {content}")

Stream mode: updates

[llm_call] returned:
  ai: tool_calls=[{'name': 'get_weather', 'args': {'city': 'London'}, 'id': 'call_nFvPq8c6czIa68OPD4kd5M31', 'type': 'tool_call'}]

[tool_node] returned:
  tool: The weather in London is 72°F and sunny.

[llm_call] returned:
  ai: It's 72°F and sunny in London.


### Stream Mode: Values

`stream_mode="values"` yields the **full state** after each node. More verbose but useful for debugging when you need to see the complete picture at each step.

In [23]:
# Stream mode: "values" - Full state after each node
print("Stream mode: values")
print("="*50)

config = {"configurable": {"thread_id": "stream-demo-2"}}

for i, state in enumerate(agent.stream(
    {"messages": [HumanMessage(content="Add 5 and 3")]},
    config=config,
    stream_mode="values"
)):
    print(f"\n[State {i}] {len(state['messages'])} messages")
    print(f"  Latest: {state['messages'][-1].type}")

Stream mode: values

[State 0] 1 messages
  Latest: human

[State 1] 2 messages
  Latest: ai

[State 2] 3 messages
  Latest: tool

[State 3] 4 messages
  Latest: ai


### Stream Mode: Messages (Token-by-Token)

`stream_mode="messages"` yields LLM output **token by token** as `(chunk, metadata)` tuples. Perfect for chat UIs that show a typing indicator. The text appears progressively as the LLM generates it.

In [24]:
# Stream mode: "messages" - Token-by-token streaming
# Perfect for chat interfaces that show typing indicators

print("Stream mode: messages (token-by-token)")
print("="*50)

config = {"configurable": {"thread_id": "stream-demo-3"}}

for message_chunk, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me a very short joke.")]},
    config=config,
    stream_mode="messages"
):
    # Only print content tokens (not tool calls)
    if message_chunk.content:
        print(message_chunk.content, end="", flush=True)

print()  # Newline at end

Stream mode: messages (token-by-token)
I told a chemistry joke — there was no reaction.


---
# Summary

This notebook covered LangGraph's advanced features:

| Feature | Purpose | Key Classes |
|---------|---------|-------------|
| **Checkpointer** | Short-term memory (per-thread) | `InMemorySaver`, `PostgresSaver` |
| **Interrupts** | Human-in-the-loop | `interrupt()`, `Command(resume=...)` |
| **Store** | Long-term memory (cross-thread) | `InMemoryStore`, `PostgresStore` |
| **Semantic Memory** | User-specific facts | `(user_id, "facts")` namespace |
| **Episodic Memory** | Agent learnings with embeddings | `store.search(query=...)` |
| **De-duplication** | Avoid duplicate learnings | Check existing before storing |
| **Streaming** | Real-time execution updates | `stream_mode="updates|values|messages"` |

### Production Considerations

- Use **database-backed** checkpointers and stores (Postgres, Redis) for persistence
- Set up **LangSmith** for tracing and debugging
- Use **`interrupt_before`/`interrupt_after`** for debugging without code changes
- Consider **message trimming** or **summarization** for long conversations

### Related Concepts Not Covered

- **Subgraphs**: Nest graphs for complex workflows
- **Middleware**: Add cross-cutting concerns (logging, caching, guardrails)
- **LangGraph Cloud**: Deploy and scale your agents
